# TriGuard — CLIP ViT-B/32 embeddings + Logistic Regression (offer-letter images: genuine vs scam)
Colab GPU runtime. Upload a zip of `data/images/` (folders `genuine/` and `scam/`) plus `image_manifest.csv`
(from `python -m ml.image.prepare_images`). Output: `image_clip.zip` -> unzip into `models/image_clip/`.
`ml/image/visual.py` loads it automatically; until then the visual score has weight 0.

In [ ]:
!pip -q install open_clip_torch scikit-learn pillow joblib
from google.colab import files
up = files.upload()   # images.zip (contains genuine/ and scam/), image_manifest.csv
!unzip -qo images.zip -d images

In [ ]:
import os, numpy as np, pandas as pd, torch, joblib, json, open_clip
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
SEED = 42
m = pd.read_csv("image_manifest.csv")
# manifest paths look like data/images/scam/x.png -> images/scam/x.png
m["path"] = m["path"].map(lambda p: os.path.join("images", *p.replace("\\", "/").split("/")[-2:]))
dev = "cuda" if torch.cuda.is_available() else "cpu"
model, _, pre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model = model.to(dev).eval()

def embed(paths, bs=64):
    out = []
    for i in range(0, len(paths), bs):
        x = torch.stack([pre(Image.open(p).convert("RGB")) for p in paths[i:i+bs]]).to(dev)
        with torch.no_grad():
            e = model.encode_image(x); e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.cpu().numpy())
    return np.concatenate(out)

X = {s: embed(m[m.split == s].path.tolist()) for s in ("train", "val", "test")}
y = {s: m[m.split == s].label.to_numpy() for s in X}

In [ ]:
clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=SEED).fit(X["train"], y["train"])
logit = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6))).reshape(-1, 1)

class Platt:
    # same interface as ml.common.calibration.PlattCalibrator, fitted on VALIDATION only
    def __init__(self, raw, y): self.lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw), y)
    def predict(self, raw): return self.lr.predict_proba(logit(np.atleast_1d(raw)))[:, 1]

pv = clf.predict_proba(X["val"])[:, 1]; cal = Platt(pv, y["val"]); cv = cal.predict(pv)
grid = np.linspace(0.01, 0.99, 99); thr = float(grid[int(np.argmax([f1_score(y["val"], cv >= t) for t in grid]))])
ct = cal.predict(clf.predict_proba(X["test"])[:, 1])
print("test F1", f1_score(y["test"], ct >= thr), "PR-AUC", average_precision_score(y["test"], ct), "ROC-AUC", roc_auc_score(y["test"], ct))

The saved joblib stores the calibrator as `ml.common.calibration.PlattCalibrator`. To keep it loadable we rebuild one
here (requires the repo on the path) — if you prefer, `pip install` nothing and instead copy `ml/common/calibration.py` next to the notebook.

In [ ]:
import sys, types, importlib
# make `ml.common.calibration.PlattCalibrator` importable in Colab (paste the class file or upload the repo's ml/ folder first)
from ml.common.calibration import PlattCalibrator
pc = PlattCalibrator().fit(pv, y["val"])
os.makedirs("image_clip", exist_ok=True)
joblib.dump({"clf": clf, "calibrator": pc, "threshold": thr, "model_name": "ViT-B-32", "pretrained": "openai"}, "image_clip/model.joblib")
!zip -qr image_clip.zip image_clip
files.download("image_clip.zip")